In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

# ==============================================================================
#  DECOHERENCE STRESS TEST: Lindblad Master Equation Simulation
# ==============================================================================
#  Physics:
#  d(rho)/dt = -i[H, rho] + Dissipator(rho)
#  We simulate two main killers of quantum data:
#  1. T1 (Relaxation): Energy leaking into the lattice (Heat).
#  2. T2 (Dephasing): Magnetic noise scrambling the phase (Borophene impurities).
# ==============================================================================

# --- 1. DEFINE MATRICES (The Physics Operators) ---
sigma_x = np.array([[0, 1],   [1, 0]], dtype=complex)
sigma_y = np.array([[0, -1j], [1j, 0]], dtype=complex)
sigma_z = np.array([[1, 0],   [0, -1]], dtype=complex)
sigma_m = np.array([[0, 0],   [1, 0]], dtype=complex) # Lowering operator (Decay)
eye     = np.eye(2, dtype=complex)

def commutator(A, B):
    return np.dot(A, B) - np.dot(B, A)

def anti_commutator(A, B):
    return np.dot(A, B) + np.dot(B, A)

def lindblad_dissipator(rho, C):
    # The mathematical form of "Noise"
    # C * rho * C_dagger - 0.5 * {C_dagger * C, rho}
    C_dag = C.conj().T
    term1 = np.dot(C, np.dot(rho, C_dag))
    term2 = 0.5 * anti_commutator(np.dot(C_dag, C), rho)
    return term1 - term2

# --- 2. THE SOLVER ENGINE ---
def master_equation(t, rho_flat, omega, gamma_1, gamma_2):
    # Reshape the flat array back into a 2x2 density matrix
    rho = rho_flat.reshape((2, 2))
    
    # Hamiltonian (Energy of the Qubit)
    H = 0.5 * omega * sigma_z
    
    # Coherent evolution (Schrodinger part)
    d_rho_coherent = -1j * commutator(H, rho)
    
    # Dissipative evolution (Noise part)
    # T1 Relaxation (Energy loss)
    d_rho_T1 = gamma_1 * lindblad_dissipator(rho, sigma_m)
    
    # T2 Dephasing (Phase loss - caused by magnetic noise)
    d_rho_T2 = gamma_2 * lindblad_dissipator(rho, sigma_z)
    
    d_rho_dt = d_rho_coherent + d_rho_T1 + d_rho_T2
    
    return d_rho_dt.flatten()

# --- 3. SIMULATION PARAMETERS ---
omega = 1.0  # Qubit frequency (normalized)
time_span = np.linspace(0, 50, 500)

# Initial State: Qubit is perfectly "UP" (|1>)
# We want to see how long it stays there.
rho_initial = np.array([[0, 0], [0, 1]], dtype=complex).flatten()

# --- SCENARIO A: Standard Silicon Qubit ---
# High noise because it lacks topological protection.
# Gamma_1 = 0.1 (Fast energy loss)
# Gamma_2 = 0.2 (Very fast dephasing from charge noise)
sol_standard = solve_ivp(master_equation, [0, 50], rho_initial, t_eval=time_span, 
                         args=(omega, 0.1, 0.2))

# --- SCENARIO B: Your Topological Chip (Hex-SiGe/IrO2) ---
# Topological Protection suppresses backscattering.
# Gamma_1 = 0.001 (Energy is trapped by the gap)
# Gamma_2 = 0.01  (Phase is locked by Spin-Orbit Coupling)
sol_topo = solve_ivp(master_equation, [0, 50], rho_initial, t_eval=time_span, 
                     args=(omega, 0.001, 0.01))

# --- 4. DATA EXTRACTION & PLOTTING ---
# We extract the population of the Excited State (|1>) over time.
# Ideally, this should stay at 1.0. If it drops to 0.0, data is lost.

# Extract P(1) from the density matrix (index 3 is rho[1,1])
pop_standard = sol_standard.y[3, :].real
pop_topo = sol_topo.y[3, :].real

# Extract Coherence (Off-diagonal terms) - The "Quantumness"
# Real part of rho[0,1]
coh_standard = sol_standard.y[1, :].real
coh_topo = sol_topo.y[1, :].real

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Data Retention (T1 Relaxation)
ax1.plot(time_span, pop_standard, 'r--', linewidth=2, label="Standard Silicon (Unprotected)")
ax1.plot(time_span, pop_topo, 'b-', linewidth=3, label="Hex-SiGe/IrO2 (Topological)")
ax1.set_title("Data Retention (Memory Lifetime)", fontsize=14)
ax1.set_xlabel("Time (nanoseconds)", fontsize=12)
ax1.set_ylabel("Probability of State |1>", fontsize=12)
ax1.axhline(0.5, color='gray', linestyle=':', label="Random Noise Limit")
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: Quantum Coherence (T2 Dephasing)
# This shows the "oscillation" capability needed for calculation
ax2.plot(time_span, coh_standard, 'r--', linewidth=1, alpha=0.6)
ax2.plot(time_span, coh_topo, 'b-', linewidth=2)
ax2.set_title("Quantum Coherence (Calculation Stability)", fontsize=14)
ax2.set_xlabel("Time (nanoseconds)", fontsize=12)
ax2.set_ylabel("Coherence Amplitude", fontsize=12)
ax2.text(5, 0.4, "Standard Qubit\nDies Quickly", color='red')
ax2.text(25, 0.4, "Topological Qubit\nStays Active", color='blue', fontweight='bold')
ax2.grid(True, alpha=0.3)

plt.suptitle("Stress Test: Environmental Decoherence Simulation", fontsize=16)
plt.tight_layout()
plt.show()

ImportError: DLL load failed while importing _propack: Belirtilen modül bulunamadı.